In [1]:
import numpy as np
import ROOT
from concurrent.futures import ProcessPoolExecutor, as_completed

In [2]:
f = ROOT.TFile.Open("/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root")
dir = f.Get("trackingNtuple")
t = dir.Get("tree")

In [5]:
import numpy as np
import ROOT
from concurrent.futures import ProcessPoolExecutor, as_completed

# ----------------------------
# User knobs
# ----------------------------
ACCEPT_RATIO = 0.75
N_WORKERS = 10
CHUNK = 100
PROGRESS_EVERY = 200  # print every ~200 entries globally

# ----------------------------
# Resolve file + tree name from existing t
# ----------------------------
root_path = t.GetCurrentFile().GetName()
tree_name = t.GetName()

print("ROOT file:", root_path)
print("Tree name:", tree_name)

def find_ttree_in_file(f, target_name):
    """
    Robustly find a TTree named `target_name` anywhere in ROOT file `f`,
    including inside nested TDirectoryFile objects.
    """
    # 1) Try direct
    obj = f.Get(target_name)
    if obj and obj.InheritsFrom("TTree"):
        return obj

    # 2) Recursive search
    def _search_dir(d):
        keys = d.GetListOfKeys()
        if not keys:
            return None
        for key in keys:
            name = key.GetName()
            obj = d.Get(name)
            if not obj:
                continue
            if obj.InheritsFrom("TTree") and obj.GetName() == target_name:
                return obj
            if obj.InheritsFrom("TDirectory"):
                found = _search_dir(obj)
                if found:
                    return found
        return None

    return _search_dir(f)

def process_range(args):
    """
    Worker: open file/tree locally, process [start, end) entries.
    Returns arrays + number of entries processed.
    """
    root_path, tree_name, start, end, accept_ratio = args

    f = ROOT.TFile.Open(root_path)
    if not f or f.IsZombie():
        raise RuntimeError(f"Could not open ROOT file: {root_path}")

    tt = find_ttree_in_file(f, tree_name)
    if not tt:
        # Help debug: list top-level keys
        top_keys = []
        lk = f.GetListOfKeys()
        if lk:
            for k in lk:
                top_keys.append(k.GetName())
        f.Close()
        raise RuntimeError(
            f"Could not find TTree '{tree_name}' anywhere in {root_path}. "
            f"Top-level keys: {top_keys[:30]}"
        )

    n = end - start
    out_event = np.empty(n, dtype=np.int64)
    out_match = np.empty(n, dtype=np.float32)
    out_fake  = np.empty(n, dtype=np.float32)

    k = 0
    for idx in range(start, end):
        tt.GetEntry(idx)

        trk_pt     = tt.trk_pt
        j          = tt.trk_bestSimTrkIdx
        share_frac = tt.trk_bestSimTrkShareFrac

        N_reco = trk_pt.size()
        if N_reco == 0:
            frac = 0.0
        else:
            N_matched = 0
            for i in range(N_reco):
                if j[i] >= 0 and share_frac[i] >= accept_ratio:
                    N_matched += 1
            frac = N_matched / N_reco

        out_event[k] = int(tt.event)
        out_match[k] = frac
        out_fake[k]  = 1.0 - frac
        k += 1

    f.Close()
    return out_event, out_match, out_fake, n

# ----------------------------
# Build chunk ranges
# ----------------------------
n_entries = int(t.GetEntries())
ranges = [
    (root_path, tree_name, s, min(s + CHUNK, n_entries), ACCEPT_RATIO)
    for s in range(0, n_entries, CHUNK)
]

print(f"Total entries: {n_entries}")
print(f"Chunks: {len(ranges)} (chunk size {CHUNK}), workers: {N_WORKERS}")

# ----------------------------
# Run in parallel with global progress printing
# ----------------------------
all_event, all_match, all_fake = [], [], []
done = 0
last_print = 0

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(process_range, r) for r in ranges]

    for fut in as_completed(futures):
        ev, mf, fr, processed = fut.result()

        all_event.append(ev)
        all_match.append(mf)
        all_fake.append(fr)

        done += processed
        if done - last_print >= PROGRESS_EVERY or done == n_entries:
            pct = 100.0 * done / n_entries if n_entries > 0 else 100.0
            print(f"{done}/{n_entries} entries processed ({pct:.1f}%)")
            last_print = done

# ----------------------------
# Concatenate + sort (by event id)
# ----------------------------
event_id   = np.concatenate(all_event)
match_frac = np.concatenate(all_match)
fake_rate  = np.concatenate(all_fake)

order = np.argsort(event_id)
event_id   = event_id[order]
match_frac = match_frac[order]
fake_rate  = fake_rate[order]

# ----------------------------
# Save cache
# ----------------------------
np.savez(
    "mapping_sanity_cache_paralell.npz",
    event_id=event_id,
    match_frac=match_frac,
    fake_rate=fake_rate,
)

print("Saved locally to mapping_sanity_cache.npz")

ROOT file: /data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root
Tree name: tree
Total entries: 1000
Chunks: 10 (chunk size 100), workers: 10
200/1000 entries processed (20.0%)
400/1000 entries processed (40.0%)
600/1000 entries processed (60.0%)
800/1000 entries processed (80.0%)
1000/1000 entries processed (100.0%)
Saved locally to mapping_sanity_cache.npz


In [ ]:
ACCEPT_RATIO = 0.75
N_WORKERS = 10
CHUNK = 1000

# You need the file + tree name so each worker can reopen them
# If you created `t` from a file, this usually works:
root_path = t.GetCurrentFile().GetName()
tree_name = t.GetName()

def process_range(args):
    """Worker: open file/tree locally, process [start, end) entries."""
    root_path, tree_name, start, end, accept_ratio = args

    f = ROOT.TFile.Open(root_path)
    tt = f.Get(tree_name)

    out_event = np.empty(end - start, dtype=np.int64)
    out_match = np.empty(end - start, dtype=np.float32)
    out_fake  = np.empty(end - start, dtype=np.float32)

    k = 0
    for idx in range(start, end):
        tt.GetEntry(idx)

        trk_pt     = tt.trk_pt
        j          = tt.trk_bestSimTrkIdx
        share_frac = tt.trk_bestSimTrkShareFrac

        N_reco = trk_pt.size()
        if N_reco == 0:
            frac = 0.0
        else:
            # loop in C++ vectors, but in this worker process
            N_matched = 0
            for i in range(N_reco):
                if j[i] >= 0 and share_frac[i] >= accept_ratio:
                    N_matched += 1
            frac = N_matched / N_reco

        out_event[k] = int(tt.event)
        out_match[k] = frac
        out_fake[k]  = 1.0 - frac
        k += 1

    f.Close()
    return out_event, out_match, out_fake, (end - start)


# Build chunk ranges
n_entries = t.GetEntries()
ranges = [(root_path, tree_name, s, min(s + CHUNK, n_entries), ACCEPT_RATIO)
          for s in range(0, n_entries, CHUNK)]

# Run in parallel
all_event, all_match, all_fake = [], [], []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(process_range, r) for r in ranges]
    for fut in as_completed(futures):
        ev, mf, fr = fut.result()
        all_event.append(ev)
        all_match.append(mf)
        all_fake.append(fr)

event_id   = np.concatenate(all_event)
match_frac = np.concatenate(all_match)
fake_rate  = np.concatenate(all_fake)

# Optional: sort results by event_id (or by original entry index if you prefer)
order = np.argsort(event_id)
event_id   = event_id[order]
match_frac = match_frac[order]
fake_rate  = fake_rate[order]

np.savez("mapping_sanity_cache.npz",
         event_id=event_id,
         match_frac=match_frac,
         fake_rate=fake_rate)

print("Saved locally to mapping_sanity_cache.npz")